# Sionna OFDM grid baseline on Google Colab

This notebook runs the Sionna resource-grid channel-estimation baseline for
[`sionna-neural-ofdm-channel-estimation`](https://github.com/s6ii5vii/sionna-neural-ofdm-channel-estimation)
on a Colab GPU. It clones the `vibrant-ptolemy-kcxqtu` branch, installs the ml
stack, verifies that Sionna's `sionna.phy` API imports, runs the test suite,
and executes the grid least-squares baseline sweep.

**Before running:** set the runtime to a GPU via
`Runtime -> Change runtime type -> T4 GPU -> Save`.

Colab runtimes are ephemeral: anything generated here disappears when the
session recycles. The final (optional) cell commits results back to the branch.

## 1. Confirm the GPU

In [ ]:
import torch
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

## 2. Clone the repository (branch `vibrant-ptolemy-kcxqtu`)

If the repository is public, run the cell as-is. If it is private, store a
fine-grained personal access token in Colab Secrets (the key icon in the left
sidebar) under the name `GH_TOKEN` and use the commented lines instead of the
plain clone. Do not paste a token directly into a cell.

In [ ]:
REPO = "s6ii5vii/sionna-neural-ofdm-channel-estimation"
BRANCH = "vibrant-ptolemy-kcxqtu"

# public repository:
!git clone --branch {BRANCH} --single-branch https://github.com/{REPO}.git

# private repository (uncomment; requires a GH_TOKEN Colab secret):
# from google.colab import userdata
# tok = userdata.get("GH_TOKEN")
# !git clone --branch {BRANCH} --single-branch https://{tok}@github.com/{REPO}.git

%cd sionna-neural-ofdm-channel-estimation

## 3. Install the ml stack

Sionna 2.x uses PyTorch and requires `numpy>=2.2.6`; installing it may change
versions of packages Colab preinstalls (a numba/numpy resolver warning is
harmless). If the import cell below fails, use `Runtime -> Restart session` once,
then re-run from this install cell onward (skip the clone).

In [ ]:
!pip install -q -e ".[ml,test]"

In [ ]:
import sionna, torch
print("sionna", sionna.__version__, "| torch", torch.__version__)

# api smoke test: these are the symbols the simulation code depends on.
from sionna.phy import config
from sionna.phy.ofdm import ResourceGrid, ResourceGridMapper, LSChannelEstimator
from sionna.phy.channel import ApplyOFDMChannel, GenerateOFDMChannel
from sionna.phy.channel.tr38901 import TDL
from sionna.phy.mapping import QAMSource
print("sionna.phy imports OK")

## 4. Run the test suite

The two tests that skip without the ml stack (`test-sionna-ofdm.py` and
`test-models.py`) should now actually run.

In [ ]:
!python -m pytest -q

## 5. Run the grid least-squares baseline

Simulates a full OFDM resource grid over a 3GPP TDL-A channel and compares
least-squares estimation with nearest and linear interpolation across the whole
grid.

In [ ]:
!python experiments/grid-tdl-v1/run-experiment.py

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv("results/tables/grid-tdl-v1.csv"))
display(Image("results/figures/grid-tdl-v1-nmse.png"))

## 6. (Optional) Generate a grid dataset

Regenerates the dataset as full OFDM resource grids and saves it. Train on it
with `python -m channel_estimation.train <a grid training config>`; training
selects the PyTorch grid CNN automatically from the 4D dataset rank and writes a
`*.report.json` with test NMSE, parameter count, serialized size, and latency.

In [ ]:
from channel_estimation.dataset import generate_grid_dataset, save_npz_dataset
from channel_estimation.sionna_ofdm import GridSpec

spec = GridSpec(
    num_subcarriers=72,
    num_ofdm_symbols=14,
    pilot_ofdm_symbol_indices=(2, 11),
    channel_kind="tdl-a",
    delay_spread_ns=100.0,
    max_doppler_hz=0.0,
)
dataset = generate_grid_dataset(spec, num_samples=2000, snr_db=10.0, random_seed=42)
save_npz_dataset("data/channel-estimation-grid-dataset.npz", dataset)
print({k: v.shape for k, v in dataset.items()})

## 7. (Optional) Commit results back to the branch

Colab is ephemeral. Run this to push generated artifacts (and any regenerated
dataset) back to the branch. Requires the `GH_TOKEN` secret configured in step 2
and push access to the repository. Note the repo's `.gitignore` excludes figures,
tables, and checkpoints by default; adjust with `git add -f` if you want to keep
specific artifacts.

In [ ]:
from google.colab import userdata

tok = userdata.get("GH_TOKEN")
!git config user.email "you@example.com"
!git config user.name "Your Name"
!git remote set-url origin https://{tok}@github.com/{REPO}.git

# example: keep the regenerated grid dataset (forced past .gitignore).
!git add -f data/channel-estimation-grid-dataset.npz
!git commit -m "add sionna grid dataset generated on colab"
!git push origin {BRANCH}